In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
import polars as pl
from datetime import timedelta, datetime, date
import numpy as np
from functools import partial

# Utils

In [ ]:
date_end = date(2026, 2, 1)
date_cl = pl.LazyFrame(
    {"date": pl.date_range(date(2025, 1, 1), date_end, interval="1d", eager=True)}
)
uid_cl = pl.LazyFrame({"uid": pl.int_range(0, 10000, eager=True)})
lf = date_cl.join(uid_cl, how="cross")


def rand_list(_, min_len, max_len) -> list:
    return list(np.random.randint(0, 10, np.random.randint(low=min_len, high=max_len)))


def create_daily_agg_pf(
    start: date,
    end: date,
    min_types_per_uid: int = 1,
    max_types_per_uid: int = 4,
    num_uids: int = 100,
) -> pl.DataFrame:
    date = pl.DataFrame({"date": pl.date_range(start, end, interval="1d", eager=True)})
    uid = pl.DataFrame({"uid": pl.int_range(0, num_uids, eager=True)})
    daily_agg_df = date.join(uid, how="cross")
    del date
    del uid
    # Add nested list of random integers
    daily_agg_df = daily_agg_df.with_columns(
        val_type=pl.col("uid").map_elements(
            function=partial(
                rand_list, min_len=min_types_per_uid, max_len=max_types_per_uid
            ),
            return_dtype=pl.List(pl.Int8),
        )
    )
    daily_agg_df = daily_agg_df.explode("val_type")
    daily_agg_df = daily_agg_pf.with_columns(
        val=pl.int_range(100, 10000).sample(n=pl.len(), with_replacement=True)
    )
    return daily_agg_df

# Initialize sintatic data

In [ ]:
daily_agg_pf = create_daily_agg_pf(
    start=date(2025, 1, 1),
    end=date(2025, 4, 1),
    min_types_per_uid=2,
    max_types_per_uid=5,
)

# Profiling

In [ ]:
daily_agg_pf.head()

In [ ]:
daily_agg_pf.filter((pl.col("date") == date(2025, 3, 15)) & (pl.col("uid") == 2))

In [ ]:
daily_agg_pf.estimated_size(unit="gb")

In [ ]:
daily_agg_pf.head()

# Expression agg context

In [ ]:
def agg_in_w_exprs(period: str) -> list[pl.Expr]:
    return [
        pl.sum("val").alias(f"sum_val_by_type_in_{period}"),
        pl.min("val").alias(f"min_val_by_type_in_{period}"),
        pl.max("val").alias(f"max_val_by_type_in_{period}"),
    ]


def agg_expr() -> list[pl.Expr]:
    return [
        pl.sum("val").alias("sum_val"),
        pl.min("val").alias("min_val"),
        pl.max("val").alias("max_val"),
    ]

# Window features backfilling context

In [ ]:
daily_agg_lf = daily_agg_pf.lazy()

In [ ]:
daily_agg_lf

## Multi-periods

In [ ]:
def rolling_f(daily_agg_lf: pl.LazyFrame, period: str) -> pl.LazyFrame:
    return daily_agg_lf.rolling(
        index_column="date", period=period, group_by=["uid", "val_type"]
    ).agg(agg_in_w_exprs(period=period))

In [ ]:
# val_type context
lf_w_3d = rolling_f(daily_agg_lf=daily_agg_lf, period="3d")
lf_w_1w = rolling_f(daily_agg_lf=daily_agg_lf, period="1w")

In [ ]:
res = lf_w_3d.collect()

## Profiling results

In [ ]:
res.head()

# Daily context

In [ ]:
def daily_f(
    daily_agg_lf: pl.LazyFrame, current_date: date, period: int
) -> pl.LazyFrame:
    period_dt = timedelta(days=period)
    return (
        daily_agg_lf.filter(
            (current_date - period_dt < pl.col("date"))
            & (pl.col("date") <= current_date)
        )
        .group_by(["uid", "val_type"])
        .agg(agg_in_w_exprs(period=f"{period}"))
    )


def daily_f_multi_period(
    daily_agg_lf: pl.LazyFrame, current_date: date, periods: list[int]
) -> pl.DataFrame:
    daily_f_all: pl.DataFrame = pl.DataFrame()
    for period in periods:
        if daily_f_all.is_empty():
            daily_f_all = daily_f(
                daily_agg_lf=daily_agg_lf, current_date=current_date, period=period
            ).collect()
        else:
            daily_f_all = daily_f_all.join(
                daily_f(
                    daily_agg_lf=daily_agg_lf, current_date=current_date, period=period
                ).collect(),
                on=["uid", "val_type"],
                how="full",
                coalesce=True,
            )
    return daily_f_all


# legacy code
def daily_f_multi_period_delta(
    daily_agg_lf: pl.LazyFrame, current_date: date, periods: list[int]
) -> pl.DataFrame:

    daily_agg_lf_prepared = daily_agg_lf.with_columns(
        (current_date - pl.col("date")).alias("delta")
    )

    group_by_period_columns = []
    for period in periods:
        col_name = f"period_{period}"
        group_by_period_columns.append(col_name)
        daily_agg_lf_prepared = daily_agg_lf_prepared.with_columns(
            (pl.col("delta").dt.total_days() // period).alias(col_name)
        ).filter(pl.col(col_name) == 0)

    # return daily_agg_lf_prepared

    return (
        daily_agg_lf_prepared.group_by(group_by_period_columns + ["uid", "val_type"])
        .agg(agg_expr())
        .collect()
    )

In [ ]:
3 // 7

In [ ]:
df = pl.DataFrame()

In [ ]:
df.is_empty()

In [ ]:
current_date = date(2025, 1, 24)

In [ ]:
daily_f_filter = daily_f_multi_period(
    daily_agg_lf=daily_agg_lf, current_date=current_date, periods=[3, 7]
)

In [ ]:
daily_f_filter.head()

In [ ]:
daily_agg_lf_prepared = daily_f_multi_period_delta(
    daily_agg_lf=daily_agg_lf, current_date=current_date, periods=[3, 7]
)

In [ ]:
daily_agg_lf_prepared.head()

In [ ]:
one_day_result_3_pf = daily_f(
    daily_agg_lf=daily_agg_lf, current_date=current_date, period=3
).collect()
one_day_result_7_pf = daily_f(
    daily_agg_lf=daily_agg_lf, current_date=current_date, period=7
).collect()

In [ ]:
one_day_result = one_day_result_3_pf.join(
    one_day_result_7_pf, on=["uid", "val_type"], how="full", coalesce=True
)

In [ ]:
one_day_result_3_pf.select(pl.col("uid").len())

In [ ]:
one_day_result.head()

In [ ]:
frame_uid_type = one_day_result_3_pf.select(pl.col("uid").unique()).join(
    one_day_result_3_pf.select(pl.col("val_type").unique()), how="cross"
)

In [ ]:
frame_uid_type.head()

In [ ]:
one_day_result_3_pf.select(pl.col("val_type").unique()).head()

In [ ]:
one_day_result_7_pf.head()